# AI Digest v2 — Course-Aligned ADK Pipeline

**Kaggle AI Agents Capstone — refactor of `ai-digest-single-agent-v1.ipynb`**

v1 used a single ADK agent that decided for itself when to call a discovery tool,
then improved its own output through a self-driven curator/critic loop. That is a
valid and arguably more advanced agentic pattern, but it doesn't map cleanly onto
the **explicit pipeline architecture** taught in the ADK course, where the developer
wires named agents together into a `SequentialAgent` / `LoopAgent` workflow and data
is passed between stages via **session state**, not by the model choosing what to do next.

This notebook keeps every functional guarantee from v1 (live arXiv discovery, a
deterministic offline fallback, and strict schema validation on the final brief),
but expresses the agent layer the way the course does:

```
DiscoveryStep (plain Python)
      |
      v
RefinementLoop = LoopAgent([CuratorAgent, CriticAgent], max_iterations=N)
      |
      v
AIDigestPipeline = SequentialAgent([RefinementLoop])
```

- **`CuratorAgent`** (`LlmAgent`) reads candidate stories from session state and writes
  a draft brief to `state['draft_brief']`.
- **`CriticAgent`** (`LlmAgent`) reads the draft, and either calls an `exit_loop` tool
  to approve it, or writes issues to `state['critic_feedback']` for the next iteration.
- **`LoopAgent`** runs the two in sequence, up to `MAX_REVISIONS` times, stopping early
  the moment `exit_loop` is called — this is the standard ADK "Generator-Critic" pattern.
- **`SequentialAgent`** is the top-level pipeline the course pattern expects; today it has
  one stage, but new stages (e.g. a publishing/notification agent) can be appended without
  touching the agents already defined.

> Runs end-to-end with a `GEMINI_API_KEY` on Kaggle, exactly like v1. Without one (or
> without network), it degrades to the same deterministic keyword fallback and still
> emits a schema-valid 10-card brief.


## Setup

Identical to v1: install/import `google.adk` + `google.genai`, load the API key from
Kaggle Secrets or the environment, and initialize a single shared `genai` client.


In [ ]:
# ── Setup: install ADK + genai, load key, init Gemini client ─────────────────
# On Kaggle this installs the real packages; locally it is a no-op if present.
try:
    import google.adk  # noqa: F401
    import google.genai  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "google-adk", "google-genai", "nest-asyncio"],
        check=False,
    )

import os
import json

# gemini-flash-latest is the current, verified-working alias for this key.
GEMINI_MODEL = "gemini-flash-latest"


def _load_api_key() -> str:
    """Load GEMINI_API_KEY from Kaggle Secrets, then the environment."""
    try:
        from kaggle_secrets import UserSecretsClient
        key = (UserSecretsClient().get_secret("GEMINI_API_KEY") or "").strip()
        if key:
            print("✅ Loaded GEMINI_API_KEY from Kaggle Secrets")
            return key
        print("⚠️  Kaggle secret 'GEMINI_API_KEY' is attached but empty")
    except ImportError:
        pass  # Not on Kaggle — fall through to the environment.
    except Exception as e:
        print(f"⚠️  Kaggle Secrets lookup failed: {type(e).__name__}: {str(e)[:120]}")
        print("   → In the Kaggle editor: attach the secret, then use "
              "'Run All' / restart the session so it is loaded.")
    key = os.getenv("GEMINI_API_KEY", "").strip()
    print("✅ Loaded GEMINI_API_KEY from environment" if key
          else "⚠️  No GEMINI_API_KEY found — will use keyword fallback")
    return key


GEMINI_API_KEY = _load_api_key()
# ADK reads GOOGLE_API_KEY; genai client reads the key we pass explicitly.
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ.setdefault("GOOGLE_GENAI_USE_ENTERPRISE", "FALSE")

# Single genai client, shared by every LLM call in this notebook.
genai_client = None
if GEMINI_API_KEY:
    try:
        from google import genai
        genai_client = genai.Client(api_key=GEMINI_API_KEY)
    except Exception as e:
        print(f"⚠️  Could not init genai client: {str(e)[:80]}")

LLM_ENABLED = genai_client is not None
print(f"LLM path: {'Gemini (' + GEMINI_MODEL + ')' if LLM_ENABLED else 'keyword fallback'}")


## Data models

Unchanged from v1. Type-safe dataclasses enforce the output contract at runtime:
valid URLs, a rank in 1–10, and exactly 10 cards per brief.


In [ ]:
# ── Models (dataclasses, stdlib only) — identical to v1 ───────────────────────
from datetime import date
from typing import List
from dataclasses import dataclass


@dataclass
class NewsItem:
    source_id: str
    title: str
    url: str
    summary: str = ""

    def __post_init__(self):
        if not self.url.startswith(("http://", "https://")):
            raise ValueError(f"Invalid URL: {self.url}")


@dataclass
class BriefCard:
    rank: int
    title: str
    url: str
    why_it_matters: str

    def __post_init__(self):
        if not (1 <= self.rank <= 10):
            raise ValueError(f"Rank must be 1-10, got {self.rank}")
        if not self.url.startswith("https://"):
            raise ValueError(f"URL must be HTTPS: {self.url}")


@dataclass
class DailyBrief:
    date: str
    theme: str
    cards: List[BriefCard]
    schema_version: str = "2.0"

    def __post_init__(self):
        if len(self.cards) != 10:
            raise ValueError(f"Must have exactly 10 cards, got {len(self.cards)}")


print("✅ Models initialized")


## Pipeline Stage 1 — Discovery

**This is the key architectural change from v1.** In v1, discovery was wrapped as a
`FunctionTool` and the agent *decided* whether and when to call it. In the course-aligned
pipeline pattern, discovery is a fixed first stage that always runs before the agents
see anything — the same live arXiv `cs.AI` / `cs.LG` RSS parsing as v1, with the same
offline fallback, just no longer gated behind a model's judgement call.


In [ ]:
# ── RSS Parser + discovery (stdlib only) — identical logic to v1 ────────────
import xml.etree.ElementTree as ET
import urllib.request


def parse_rss_bytes(data: bytes, source_id: str, limit: int = 20) -> List[NewsItem]:
    """Parse RSS 2.0 / Atom feeds using stdlib only."""
    try:
        root = ET.fromstring(data)
    except ET.ParseError:
        return []

    items = []
    _ATOM = "http://www.w3.org/2005/Atom"
    _CONTENT = "{http://purl.org/rss/1.0/modules/content/}encoded"

    def _text(el):
        return (el.text or "").strip() if el is not None else ""

    channel = root.find("channel")
    if channel is not None:  # RSS 2.0
        for el in channel.findall("item"):
            title = _text(el.find("title"))
            url = _text(el.find("link"))
            summary = _text(el.find("description")) or _text(el.find(_CONTENT))
            if title and url and url.startswith("http"):
                try:
                    items.append(NewsItem(source_id, title, url, summary[:500]))
                except ValueError:
                    pass
            if len(items) >= limit:
                break
        return items

    for el in root.findall(f"{{{_ATOM}}}entry"):  # Atom
        title = _text(el.find(f"{{{_ATOM}}}title"))
        url = ""
        for link in el.findall(f"{{{_ATOM}}}link"):
            href = link.get("href", "")
            if link.get("rel", "alternate") in ("alternate", "") and href.startswith("http"):
                url = href
                break
        summary = _text(el.find(f"{{{_ATOM}}}summary")) or _text(el.find(f"{{{_ATOM}}}content"))
        if title and url:
            try:
                items.append(NewsItem(source_id, title, url, summary[:500]))
            except ValueError:
                pass
        if len(items) >= limit:
            break
    return items


def _fallback_items() -> List[NewsItem]:
    """Sample AI/ML items used only when arXiv is unreachable."""
    raw = [
        ("Llama 3.1 405B Reaches State-of-the-Art Performance", "2407.21022",
         "Meta's latest LLM shows breakthrough reasoning and multilingual gains."),
        ("Vision Transformers Show Strong Zero-Shot Transfer", "2407.20299",
         "ViT models maintain performance across diverse visual domains."),
        ("Scaling Laws for Neural Language Models Revisited", "2407.20322",
         "Compute-optimal scaling reveals new LLM training-efficiency patterns."),
        ("RAG Improves Hallucination Control", "2407.18872",
         "Retrieval-augmented generation cuts factual errors across domains."),
        ("Self-Supervised Learning Sets New Benchmark Records", "2407.19234",
         "SSL rivals supervised approaches on ImageNet."),
        ("Graph Neural Networks for Molecular Property Prediction", "2407.18765",
         "GNNs beat traditional methods on drug-discovery benchmarks."),
        ("LoRA Fine-Tuning Nears Full-Model Performance", "2407.19223",
         "Low-rank adaptation cuts memory while keeping quality."),
        ("Multimodal Transformers Excel at Vision-Language Tasks", "2407.18934",
         "End-to-end models lead on VQA and captioning."),
        ("Continuous Learning Enables Lifelong Adaptation", "2407.19012",
         "New methods prevent catastrophic forgetting."),
        ("Sparse Attention Kernels Set Efficiency Records", "2407.18823",
         "Sparse patterns cut compute without hurting performance."),
    ]
    return [NewsItem("arxiv", t, f"https://arxiv.org/abs/{i}", s) for t, i, s in raw]


def discover_news() -> List[NewsItem]:
    """Pipeline Stage 1: fetch fresh AI/ML papers, with an offline fallback.

    Always runs first, unconditionally — this is what makes it a pipeline stage
    rather than a tool the model opts into calling.
    """
    sources = [
        ("arxiv-cs-ai", "https://arxiv.org/rss/cs.AI"),
        ("arxiv-cs-lg", "https://arxiv.org/rss/cs.LG"),
    ]
    items: List[NewsItem] = []
    for sid, url in sources:
        try:
            print(f"   {sid}...", end="", flush=True)
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=10) as resp:
                items.extend(parse_rss_bytes(resp.read(), sid, limit=20))
            print(" ✓")
        except Exception:
            print(" ✗")
    if not items:
        print("   Using offline fallback data")
        items = _fallback_items()
    return items


print("\n🔍 Stage 1/3 — Discovery\n")
discovered_items = discover_news()
print(f"\n✅ Discovered {len(discovered_items)} items")


## Reasoning primitives + deterministic fallback

Unchanged from v1: keyword-based ranking as the offline safety net, and
`assemble_brief`, which turns whatever cards an agent (or the fallback) proposes
into a schema-valid, deduplicated, exactly-10-card `DailyBrief`.


In [ ]:
# ── Reasoning primitives + deterministic fallback — identical to v1 ────────
_KEYWORDS = {
    3: ("model", "llm", "agent", "reasoning", "benchmark", "eval"),
    2: ("ai", "ml", "learning", "transformer", "multimodal"),
}


def _score(item: NewsItem) -> int:
    text = f"{item.title} {item.summary}".lower()
    return sum(pts for pts, kws in _KEYWORDS.items() if any(k in text for k in kws))


def keyword_rank(items: List[NewsItem], k: int = 10) -> List[NewsItem]:
    """Deterministic keyword ranking (offline fallback)."""
    return sorted(items, key=lambda x: (-_score(x), x.title.lower()))[:k]


def _https(url: str) -> str:
    return url.replace("http://", "https://", 1) if url.startswith("http://") else url


def assemble_brief(cards_data: list, theme: str = "AI signal over noise") -> DailyBrief:
    """Turn proposed cards into a schema-valid 10-card DailyBrief."""
    by_url = {i.url: i for i in discovered_items}
    seen, cards = set(), []
    for c in cards_data:
        title = (c.get("title") or "").strip()
        url = _https((c.get("url") or "").strip())
        if not title or not url.startswith("https://") or title.lower() in seen:
            continue
        seen.add(title.lower())
        why = (c.get("why_it_matters") or "").strip()
        if not why:
            src = by_url.get(url)
            why = (src.summary[:200] if src and src.summary else "Key AI/ML development.")
        cards.append((title, url, why))
    # Pad from keyword ranking if fewer than 10 usable cards were proposed.
    for item in keyword_rank(discovered_items, k=len(discovered_items)):
        if len(cards) >= 10:
            break
        if item.title.lower() not in seen and item.url.startswith("https://"):
            seen.add(item.title.lower())
            cards.append((item.title, item.url,
                          item.summary[:200] or "Key AI/ML development."))
    cards = cards[:10]
    return DailyBrief(
        date=str(date.today()),
        theme=theme,
        cards=[BriefCard(i + 1, t, u, w) for i, (t, u, w) in enumerate(cards)],
    )


def keyword_brief(items: List[NewsItem]) -> DailyBrief:
    """Full deterministic brief — used when the LLM pipeline is unavailable."""
    ranked = keyword_rank(items, k=10)
    return assemble_brief(
        [{"title": it.title, "url": it.url, "why_it_matters": it.summary[:200]}
         for it in ranked]
    )


def _parse_json(text: str) -> dict:
    text = (text or "").strip().replace("```json", "").replace("```", "").strip()
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1:
        text = text[start:end + 1]
    return json.loads(text)


def brief_to_dict(b: DailyBrief) -> dict:
    return {
        "date": b.date, "theme": b.theme, "schema_version": b.schema_version,
        "cards": [{"rank": c.rank, "title": c.title, "url": c.url,
                   "why_it_matters": c.why_it_matters} for c in b.cards],
    }


print("✅ Reasoning primitives + fallback ready")


## Pipeline Stages 2–3 — the ADK workflow

This is the course-style refactor. Instead of one `Agent` that owns a tool and
decides everything itself, we declare named `LlmAgent` roles and wire them into
the workflow-agent classes ADK ships specifically for this: `LoopAgent` for the
generator/critic refinement cycle, and `SequentialAgent` as the top-level pipeline.

- **`CuratorAgent`** reads `{candidate_stories}` and `{critic_feedback}` from session
  state (both are injected via `instruction` templating) and writes its draft to
  `state['draft_brief']` via `output_key`.
- **`CriticAgent`** reads `{draft_brief}`, and either calls the `exit_loop` tool to
  approve it (which escalates and stops the `LoopAgent`), or writes fix-it notes to
  `state['critic_feedback']` for the curator's next pass.
- **`RefinementLoop`** (`LoopAgent`) runs Curator → Critic up to `MAX_REVISIONS` times,
  or fewer if `exit_loop` fires early.
- **`AIDigestPipeline`** (`SequentialAgent`) is the single top-level entry point —
  today it just runs `RefinementLoop`, but new stages (e.g. a `PublisherAgent`) could
  be appended after it without touching Curator/Critic at all. That extensibility is
  the main practical benefit of the pipeline shape over v1's single free agent.


In [ ]:
# ── Real Google ADK: SequentialAgent + LoopAgent + LlmAgent ─────────────────
import uuid
import asyncio

MAX_REVISIONS = 2
ADK_AVAILABLE = False
ai_digest_pipeline = None

try:
    from google.adk.agents import LlmAgent, SequentialAgent, LoopAgent
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.adk.tools import FunctionTool
    from google.adk.tools.tool_context import ToolContext
    from google.genai.types import Content, Part

    _session_service = InMemorySessionService()
    _APP = "ai_digest_v2"
    _USER = "pipeline"

    def exit_loop(tool_context: ToolContext) -> dict:
        """Call this only when the current brief needs no further revisions.

        Calling it stops the refinement loop early instead of running the full
        MAX_REVISIONS iterations.
        """
        tool_context.actions.escalate = True
        return {"status": "approved"}

    curator_agent = LlmAgent(
        name="CuratorAgent",
        model=GEMINI_MODEL,
        description="Curates the top 10 AI/ML stories into a daily brief.",
        instruction=(
            "Candidate stories (JSON array of title/url/summary):\n"
            "{candidate_stories}\n\n"
            "Previous critic feedback, if any (empty on the first pass):\n"
            "{critic_feedback}\n\n"
            "Choose the 10 most important stories and rank them (1 = most "
            "important). Prefer frontier models, agents, benchmarks, and results "
            "with real impact. If critic feedback is present, fix every issue it "
            "raises. Return ONLY JSON:\n"
            '{"theme": "<short theme>", "cards": [{"rank": 1, "title": "...", '
            '"url": "...", "why_it_matters": "<= 30 words, specific"}]}\n'
            "Use each story's URL exactly as provided. Return exactly 10 cards."
        ),
        output_key="draft_brief",
    )

    critic_agent = LlmAgent(
        name="CriticAgent",
        model=GEMINI_MODEL,
        description="Reviews the draft brief and either approves or lists fixes.",
        instruction=(
            "Review this draft brief JSON as a strict editor:\n"
            "{draft_brief}\n\n"
            "Check for: duplicate or near-duplicate stories, vague or generic "
            "why_it_matters text, and poor importance ordering.\n\n"
            "If the brief is genuinely high quality, call the exit_loop tool and "
            "say nothing else. Otherwise, respond with ONLY JSON describing the "
            'problems: {"issues": ["..."]}.'
        ),
        tools=[FunctionTool(exit_loop)],
        output_key="critic_feedback",
    )

    refinement_loop = LoopAgent(
        name="RefinementLoop",
        sub_agents=[curator_agent, critic_agent],
        max_iterations=MAX_REVISIONS,
    )

    ai_digest_pipeline = SequentialAgent(
        name="AIDigestPipeline",
        sub_agents=[refinement_loop],
    )

    ADK_AVAILABLE = LLM_ENABLED
except Exception as e:
    print(f"⚠️  ADK pipeline unavailable ({str(e)[:70]}) — will use fallback")


async def _run_pipeline_async(candidate_stories: str) -> dict:
    sid = str(uuid.uuid4())
    initial_state = {"candidate_stories": candidate_stories, "critic_feedback": ""}
    await _session_service.create_session(
        app_name=_APP, user_id=_USER, session_id=sid, state=initial_state
    )
    runner = Runner(agent=ai_digest_pipeline, app_name=_APP, session_service=_session_service)
    message = Content(role="user", parts=[Part(text="Produce today's AI Digest brief.")])
    async for _ in runner.run_async(user_id=_USER, session_id=sid, new_message=message):
        pass  # We read the result from session state, not the event stream.
    session = await _session_service.get_session(app_name=_APP, user_id=_USER, session_id=sid)
    return session.state


def run_pipeline(candidate_stories: str) -> dict:
    """Synchronous wrapper so the pipeline runs inside the Kaggle notebook loop."""
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass
    return asyncio.get_event_loop().run_until_complete(
        _run_pipeline_async(candidate_stories)
    )


print(f"✅ ADK pipeline ready: {ADK_AVAILABLE}"
      + (" — AIDigestPipeline(RefinementLoop(CuratorAgent, CriticAgent))" if ADK_AVAILABLE else ""))


## Run — the full pipeline, stage by stage

Unlike v1, where `generate_brief()` interleaved discovery, curation, and critique
inside one function driven by an agent's own choices, here the three stages are
visibly sequential: `discovered_items` was already produced in Stage 1 above; this
cell only runs Stage 2–3 (`AIDigestPipeline`) and Stage 4 (fallback-or-not decision).
Every path still ends in a schema-valid 10-card `DailyBrief`.


In [ ]:
# ── Stage 2–3: run the ADK pipeline; Stage 4: fall back if it fails ─────────
def generate_brief_v2():
    if not ADK_AVAILABLE:
        print("→ LLM pipeline unavailable, using keyword fallback")
        return keyword_brief(discovered_items), "keyword-fallback"

    try:
        print("🤖 Stage 2/3 — running AIDigestPipeline (Curator ⇄ Critic)...")
        candidate_stories = json.dumps(
            [{"title": i.title, "url": i.url, "summary": i.summary}
             for i in discovered_items]
        )
        final_state = run_pipeline(candidate_stories)
        data = _parse_json(final_state.get("draft_brief", ""))
        brief = assemble_brief(data.get("cards", []),
                               data.get("theme", "AI signal over noise"))
        print("✅ Stage 3/3 — pipeline finished (approved or max iterations reached)")
        return brief, "adk-pipeline"
    except Exception as e:
        print(f"⚠️  ADK pipeline failed ({str(e)[:80]}), using keyword fallback")
        return keyword_brief(discovered_items), "keyword-fallback"


brief, brief_path = generate_brief_v2()
print(f"\n✅ Generated DailyBrief via '{brief_path}' ({len(brief.cards)} cards)")


## Output & validation

Unchanged from v1. Display the brief, export it as JSON, and assert the schema
contract holds — exactly 10 cards, ranks 1–10, HTTPS URLs, and non-empty explanations.


In [ ]:
# ── Display, export, validate — identical to v1 ───────────────────────────
print("=" * 70)
print(f"AI DIGEST v2 — {brief.date}  |  theme: {brief.theme}  |  via: {brief_path}")
print("=" * 70)
for c in brief.cards:
    print(f"[{c.rank:2}] {c.title}")
    print(f"     {c.url}")
    print(f"     → {c.why_it_matters}")

# Export for downstream use / Kaggle output.
out = brief_to_dict(brief)
with open("brief_output_v2.json", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print("\n💾 Wrote brief_output_v2.json")

# Schema contract — fails loudly if the pipeline ever breaks the format.
assert len(brief.cards) == 10, "must have exactly 10 cards"
assert [c.rank for c in brief.cards] == list(range(1, 11)), "ranks must be 1..10"
assert all(c.url.startswith("https://") for c in brief.cards), "URLs must be HTTPS"
assert all(c.why_it_matters.strip() for c in brief.cards), "explanations required"
assert len({c.title.lower() for c in brief.cards}) == 10, "no duplicate titles"
print("✅ Schema validation passed (10 cards, ranks 1-10, HTTPS, no dupes)")


## v1 vs v2 — what changed and why

| | v1 (single-agent) | v2 (this notebook) |
|---|---|---|
| Discovery | `FunctionTool` the agent chooses to call | Fixed Stage 1, always runs first |
| Agent shape | One `Agent` holding a tool + instructions | Named `LlmAgent` roles: `CuratorAgent`, `CriticAgent` |
| Orchestration | Hand-written Python loop calling `run_adk` twice | `SequentialAgent(LoopAgent([Curator, Critic]))` — ADK's own workflow classes |
| Data handoff | Function arguments / return values | Session **state** (`output_key`, `{template}` injection) |
| Loop exit | `if review.get("approved"): break` in application code | `CriticAgent` calls an `exit_loop` tool that escalates inside ADK |
| Extensibility | Adding a new stage means editing `generate_brief()` | Adding a new stage means appending to `AIDigestPipeline.sub_agents` |
| Data models, discovery, fallback, validation | Same | **Unchanged** |

**Trade-offs, honestly stated:**

- v1's single agent is arguably *more* agentic — the model decides for itself whether
  to fetch news at all, not just how to rank it. v2 fixes discovery as a mandatory
  first step, which is less flexible but easier to reason about and test in isolation.
- v2's explicit pipeline is closer to what the ADK course walks through step by step,
  and each stage (`LlmAgent`, `LoopAgent`, `SequentialAgent`) is independently inspectable
  and swappable — useful if a grader or teammate wants to see "where the pipeline is."
- Both notebooks keep the same non-negotiable guarantees: live arXiv discovery with an
  offline fallback, and a brief that can never leave the notebook in an invalid schema.

So the accurate one-line answer to "why did you remove the pipeline": **it wasn't removed
in v1, it was implicit** — discovery, curation, critique, and validation still happened in
a fixed overall order, but the *decision-making inside* that order (tool use, ranking,
when to stop revising) was delegated to the agent instead of hard-coded. v2 makes that
same reasoning happen inside course-standard `SequentialAgent` / `LoopAgent` wiring instead.
